# Medical Insurance Cost Prediction Using ML with Explainable AI

**Paper:** "Machine Learning for an Explainable Cost Prediction of Medical Insurance"  
**Authors:** Ugochukwu Orji & Elochukwu Ukwandu (2023)  
**Published:** Machine Learning with Applications, ScienceDirect  
**Dataset:** US Medical Cost Personal Dataset (Kaggle)  
**Student:** Anuraag Potdaar

In [ ]:
# Imports and Setup
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Project root (one level up from notebooks/)
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure output directories exist
os.makedirs(os.path.join(PROJECT_ROOT, 'results/figures'), exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'results/tables'), exist_ok=True)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All imports successful!")

In [ ]:
# Load and Preprocess Data
from src.data_preprocessing import load_and_preprocess

DATA_PATH = os.path.join(PROJECT_ROOT, 'data/insurance.csv')
X_train, X_test, y_train, y_test, df, scaler = load_and_preprocess(DATA_PATH)

print(f"\nDataset shape: {df.shape}")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeature columns: {list(X_train.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical Summary (Table 3)
from src.visualization import statistical_summary

summary = statistical_summary(df)

In [ ]:
# EDA — Correlation Heatmap (Figure 3)
from src.visualization import plot_correlation_heatmap

plot_correlation_heatmap(df)

In [ ]:
# EDA — Feature vs Charges plots
from src.visualization import plot_feature_vs_target

plot_feature_vs_target(df)

In [ ]:
# Train Models with GridSearchCV
from src.models import train_all_models

print("Training all models with GridSearchCV...")
print("This may take several minutes...\n")
results = train_all_models(X_train, y_train)
print("\nTraining complete!")

In [ ]:
# Print Training Results and Best Parameters (Table 7)
print("=" * 80)
print("TRAINING RESULTS (Table 7)")
print("=" * 80)

training_rows = []
for name, res in results.items():
    print(f"\n--- {name} ---")
    print(f"  Best Parameters: {res['best_params']}")
    print(f"  Train R²: {res['train_r2']:.3f}%")
    print(f"  CV R²:    {res['cv_r2']:.3f}%")
    print(f"  Time:     {res['time_secs']:.3f} seconds")
    print(f"  Memory:   {res['memory_mb']:.3f} MB")
    training_rows.append({
        'Model': name,
        'Train R2 (%)': round(res['train_r2'], 3),
        'CV R2 (%)': round(res['cv_r2'], 3),
        'Time (s)': round(res['time_secs'], 3),
        'Memory (MB)': round(res['memory_mb'], 3),
        'Best Parameters': str(res['best_params'])
    })

training_df = pd.DataFrame(training_rows)
training_df.to_csv(os.path.join(PROJECT_ROOT, 'results/tables/training_results.csv'), index=False)
print("\n\nTraining results saved to results/tables/training_results.csv")
training_df

In [ ]:
# Evaluate Models on Test Data (Table 9)
from src.evaluation import evaluate_all_models

metrics_df, predictions = evaluate_all_models(results, X_test, y_test)
metrics_df

In [ ]:
# Learning Curves (Figure 5)
from src.visualization import plot_learning_curves

plot_learning_curves(results, X_train, y_train)

In [ ]:
# Residual Plots (Figures 6-8)
from src.visualization import plot_residuals

plot_residuals(predictions, y_test, results)

In [ ]:
# Prediction Error Plots (Figure 10)
from src.visualization import plot_prediction_error

plot_prediction_error(predictions, y_test)

In [ ]:
# SHAP Analysis — All 3 Models (Figures 11-13)
from src.explainability import shap_analysis

feature_names = list(X_train.columns)
shap_analysis(results, X_train, X_test, feature_names)

In [ ]:
# ICE Plot Analysis — All 3 Models (Figures 14-16)
from src.explainability import ice_analysis

ice_analysis(results, X_test, feature_names)

In [ ]:
# Results Comparison Table (Tables 7, 8, 9)
from src.evaluation import compute_improvement_table

print("=" * 80)
print("TABLE 7 — Training Results")
print("=" * 80)
print(training_df[['Model', 'Train R2 (%)', 'CV R2 (%)', 'Time (s)', 'Memory (MB)']].to_string(index=False))

print("\n")
print("=" * 80)
print("TABLE 8 — Improvement After Hyperparameter Tuning")
print("=" * 80)
improvement_df = compute_improvement_table(results, X_test, y_test)

print("\n")
print("=" * 80)
print("TABLE 9 — Model Performance on Test Data")
print("=" * 80)
print(metrics_df.to_string(index=False))

# Summary and Conclusions

## Key Findings

### Model Performance
- **XGBoost** and **GBM** achieved the best R² on test data, but XGBoost consumed the most computational resources
- **GBM** provided competitive accuracy with efficient resource usage
- **Random Forest** showed strong training performance but lower generalization

### Explainability (SHAP Analysis)
- **Smoker status** is by far the most influential feature across all models
- **Age** and **BMI** are the next most important predictors
- **Region** and **sex** have relatively minor impact on predictions

### Explainability (ICE Analysis)
- **Smoker** shows a dramatic step increase in insurance charges
- **Age** and **BMI** show gradual positive relationships with cost
- **Number of children** has minimal impact on predicted charges
- **ICE plots** reveal individual-level heterogeneity beyond average SHAP effects

### Overall Conclusion
Ensemble tree-based models achieve strong performance for US medical insurance cost prediction. Explainability methods (SHAP + ICE) confirm that smoking status is the dominant cost driver, followed by age and BMI — consistent with actuarial domain knowledge.